# Paper 15 · Denoising Diffusion Probabilistic Models

**Citation:** Jonathan Ho, Ajay Jain, Pieter Abbeel, “Denoising Diffusion Probabilistic Models” (2020).

**Paper:** https://arxiv.org/abs/2006.11239

> **Scale gap:** We diffuse and regenerate a 2-D mixture distribution with a tiny MLP, not high-resolution images.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 12 · Generative-Model Mathematics](../../math/12_generative_models_math.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What information remains as diffusion time increases?
2. Why can noise prediction be a convenient training target?
3. Why does sampling require many reverse steps?

## Central claim
A generative model can be trained by learning to reverse a gradual Gaussian noising process, commonly via noise prediction.

## Forward diffusion

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-15_ddpm', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/15_ddpm.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0); np.random.seed(0)
T=100
betas=torch.linspace(1e-4,.2,T)
alphas=1-betas
abar=torch.cumprod(alphas,0)
assert abar[-1] < 1e-3, "Terminal distribution must approach the sampling prior"
abar_prev=torch.cat([torch.ones(1),abar[:-1]])
posterior_var=betas*(1-abar_prev)/(1-abar)
centers=torch.tensor([[2.,0.],[-2.,0.],[0.,2.],[0.,-2.]])
def sample_data(n):
    ids=torch.randint(0,4,(n,))
    return centers[ids]+.25*torch.randn(n,2)
def q_sample(x0,t,eps=None):
    if eps is None: eps=torch.randn_like(x0)
    a=abar[t][:,None]
    return torch.sqrt(a)*x0+torch.sqrt(1-a)*eps,eps

x0=sample_data(2000)
fig,axs=plt.subplots(1,4,figsize=(12,3))
for ax,tt in zip(axs,[0,20,60,99]):
    t=torch.full((len(x0),),tt,dtype=torch.long)
    xt,_=q_sample(x0,t)
    ax.scatter(xt[:,0],xt[:,1],s=3); ax.set_title(f"t={tt}"); ax.set_xlim(-4,4); ax.set_ylim(-4,4)
plt.tight_layout(); plt.show()

## Noise-prediction network

In [ ]:
model=nn.Sequential(nn.Linear(3,64),nn.SiLU(),nn.Linear(64,64),nn.SiLU(),nn.Linear(64,2))
opt=torch.optim.Adam(model.parameters(),lr=.002)
losses=[]
for step in range(1200):
    x0=sample_data(256); t=torch.randint(0,T,(len(x0),))
    xt,eps=q_sample(x0,t)
    inp=torch.cat([xt,(t.float()/T)[:,None]],1)
    pred=model(inp); loss=((pred-eps)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
print("final noise MSE",losses[-1])
plt.plot(losses); plt.yscale("log"); plt.title("Noise-prediction loss"); plt.show()

## Reverse sampling

In [ ]:
@torch.no_grad()
def sample_reverse(n=2000):
    x=torch.randn(n,2)
    for ti in reversed(range(T)):
        t=torch.full((n,),ti,dtype=torch.long)
        eps=model(torch.cat([x,(t.float()/T)[:,None]],1))
        mean=(x-(betas[ti]/torch.sqrt(1-abar[ti]))*eps)/torch.sqrt(alphas[ti])
        x=mean+(torch.sqrt(posterior_var[ti])*torch.randn_like(x) if ti>0 else 0)
    return x
gen=sample_reverse()
plt.scatter(gen[:,0],gen[:,1],s=3,alpha=.4); plt.xlim(-4,4); plt.ylim(-4,4); plt.title("Generated samples"); plt.show()

### Ablation
Change the number of diffusion steps and beta schedule. Compare noise-prediction loss and visual mode coverage.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this work?
2. What was actually new?
3. What evidence did this notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which contribution remains important today?
6. What would you test next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))

## Methodology note

This is a bounded mechanism experiment, not an original-benchmark reproduction. A run that completes is not evidence that the paper claim was reproduced. Report actual baseline comparisons, uncertainty and failed ablations.